# PyAerial Quickstart

PyAerial finds human-readable IF-THEN rules in tabular data using **Aerial**, a scalable neurosymbolic association rule miner: an under-complete Autoencoder learns a compact representation of the data, and rules are extracted from the trained model.

- Documentation: https://pyaerial.readthedocs.io
- Paper: [Neurosymbolic Association Rule Mining from Tabular Data](https://proceedings.mlr.press/v284/karabulut25a.html)
- GitHub: https://github.com/DiTEC-project/pyaerial

In [ ]:
%pip install pyaerial ucimlrepo

## 1. Basic association rule mining

Load a categorical tabular dataset, train the Autoencoder, and extract rules. Quality metrics (support, confidence, Zhang's metric) are calculated automatically.

In [ ]:
from aerial import model, rule_extraction
from ucimlrepo import fetch_ucirepo

# a categorical tabular dataset
breast_cancer = fetch_ucirepo(id=14).data.features
breast_cancer.head()

In [ ]:
trained_autoencoder = model.train(breast_cancer)

result = rule_extraction.generate_rules(trained_autoencoder, min_rule_frequency=0.1, min_rule_strength=0.8)

print(f"Overall statistics: {result['statistics']}\n")
print(f"Sample rule: {result['rules'][0]}")

Rules are plain dictionaries — print them in a readable IF-THEN form:

In [ ]:
for rule in result['rules'][:10]:
    antecedents = " AND ".join(f"{a['feature']}={a['value']}" for a in rule['antecedents'])
    consequent = f"{rule['consequent']['feature']}={rule['consequent']['value']}"
    print(f"IF {antecedents} THEN {consequent} (support: {rule['support']:.2f}, conf: {rule['confidence']:.2f})")

## 2. Numerical data

PyAerial works with categorical data; numerical columns are discretized first with the built-in discretization module (8 methods available).

In [ ]:
from aerial import discretization

# a numerical dataset
iris = fetch_ucirepo(id=53).data.features

iris_discretized = discretization.equal_frequency_discretization(iris, n_bins=3)

trained_autoencoder = model.train(iris_discretized, epochs=10)
result = rule_extraction.generate_rules(trained_autoencoder, min_rule_frequency=0.1)
print(f"Found {result['statistics']['rule_count']} rules with avg support {result['statistics']['average_support']:.3f}")

## 3. Classification rules

Restrict rule consequents to a class label column for interpretable classification.

In [ ]:
import pandas as pd

dataset = fetch_ucirepo(id=14)
table_with_labels = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

trained_autoencoder = model.train(table_with_labels)
result = rule_extraction.generate_rules(
    trained_autoencoder,
    target_classes=["Class"],
    min_rule_frequency=0.01,
    min_rule_strength=0.7,
)

for rule in result['rules'][:5]:
    antecedents = " AND ".join(f"{a['feature']}={a['value']}" for a in rule['antecedents'])
    print(f"IF {antecedents} THEN {rule['consequent']['value']} (conf: {rule['confidence']:.2f})")

## Next steps

- [Parameter Tuning Guide](https://pyaerial.readthedocs.io/en/latest/parameter_guide.html) — defaults, tuning for specific goals, troubleshooting
- [User Guide](https://pyaerial.readthedocs.io/en/latest/user_guide.html) — item constraints, frequent itemsets, rule visualization, and more
- [How Aerial Works](https://pyaerial.readthedocs.io/en/latest/research.html) — the neurosymbolic architecture and rule extraction algorithm